# SmartBank: BANKING77 NLP Chatbot
### Simple Google Colab version

**Pipeline:** BANKING77 → Pandas → spaCy → TF-IDF → Logistic Regression → Evaluation → Live Chatbot

This is a classroom/demo NLP project. Do not enter real passwords, PINs, OTPs, card numbers, or other sensitive banking information.


## 1. Install libraries

In [ ]:
!pip install -q pandas numpy scikit-learn spacy
!python -m spacy download en_core_web_sm


## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")


## 3. Load spaCy

In [ ]:
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully!")


## 4. Upload Kaggle BANKING77 CSV files

Upload `train.csv` and `test.csv` from your BANKING77 Kaggle dataset.

In [ ]:
from google.colab import files
uploaded = files.upload()


## 5. Read the dataset

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Training data:")
display(train_df.head())

print("\nTesting data:")
display(test_df.head())


Training data:


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival



Testing data:


,text,category
0,How do I locate my card?,card_arrival
1,"I still have not received my new card, I order...",card_arrival
2,I ordered a card but it has not arrived. Help ...,card_arrival
3,Is there a way to know when my card will arrive?,card_arrival
4,My card has not arrived yet.,card_arrival


## 6. Understand the dataset

In [ ]:
print("Training Shape:", train_df.shape)
print("Testing Shape:", test_df.shape)

print("\nTraining Columns:")
print(train_df.columns)

print("\nMissing Values:")
print(train_df.isnull().sum())


Training Shape: (10003, 2)
Testing Shape: (3080, 2)

Training Columns:
Index(['text', 'category'], dtype='object')

Missing Values:
text        0
category    0
dtype: int64


## 7. Look at real banking questions

In [ ]:
for i in range(10):
    print("Question:", train_df.iloc[i]["text"])
    print("Intent:", train_df.iloc[i]["category"])
    print("-" * 60)

## 8. Check the number of intents

In [ ]:
print("Number of unique intents:", train_df["category"].nunique())
print("\nIntent names:")
print(train_df["category"].unique())

## 9. Intent distribution

In [ ]:
intent_counts = train_df["category"].value_counts()
display(intent_counts)

## 10. Create the spaCy preprocessing function

In [ ]:
def preprocess_text(text):
    doc = nlp(text.lower())
    words = []

    for token in doc:
        if token.is_punct:
            continue
        if token.is_stop:
            continue
        if token.is_space:
            continue

        words.append(token.lemma_)

    return " ".join(words)


## 11. Test spaCy preprocessing

In [ ]:
text = "I have forgotten my password and I cannot login"
clean_text = preprocess_text(text)

print("Original:")
print(text)

print("\nAfter preprocessing:")
print(clean_text)


Original:
I have forgotten my password and I cannot login

After preprocessing:
forget password login


## 12. Preprocess training data

In [ ]:
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
train_df[["text", "clean_text", "category"]].head(10)

## 13. Preprocess test data

In [ ]:
test_df["clean_text"] = test_df["text"].apply(preprocess_text)
test_df[["text", "clean_text", "category"]].head()

## 14. Separate input and target

In [ ]:
X_train = train_df["clean_text"]
y_train = train_df["category"]

X_test = test_df["clean_text"]
y_test = test_df["category"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

## 15. Convert text into numbers using TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000
)

X_train_tfidf = tfidf.fit_transform(X_train) #scaling
X_test_tfidf = tfidf.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)


## 16. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train_tfidf, y_train)

print("Model training completed!")


Model training completed!


## 17. Make predictions

In [ ]:
y_pred = model.predict(X_test_tfidf)
print(y_pred[:20])


['country_support' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_arrival' 'getting_spare_card' 'card_arrival'
 'card_delivery_estimate' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_delivery_estimate' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_delivery_estimate' 'card_delivery_estimate' 'order_physical_card'
 'card_arrival' 'card_arrival']


## 18. Check accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", round(accuracy * 100, 2), "%")


Model Accuracy: 81.82 %


## 19. Detailed evaluation

In [ ]:
print(classification_report(y_test, y_pred))


## 20. Create the prediction function

In [ ]:
def predict_intent(message):
    clean_message = preprocess_text(message)
    message_vector = tfidf.transform([clean_message])
    prediction = model.predict(message_vector)[0]
    probabilities = model.predict_proba(message_vector)
    confidence = np.max(probabilities)

    return prediction, confidence


## 21. Test one customer question

In [ ]:
message = "My card has not arrived yet"

intent, confidence = predict_intent(message)

print("Customer:", message)
print("Predicted Intent:", intent)
print("Confidence:", round(confidence * 100, 2), "%")


Customer: My card has not arrived yet
Predicted Intent: card_arrival
Confidence: 35.02 %


## 22. Test multiple real-world questions

In [ ]:
questions = [
    "My card has not arrived",
    "I forgot my PIN",
    "Why was I charged for withdrawing cash?",
    "I don't recognize this transaction",
    "My transfer is still pending",
    "I want to change my PIN"
]

for question in questions:
    intent, confidence = predict_intent(question)

    print("Customer:", question)
    print("Intent:", intent)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 60)


Customer: My card has not arrived
Intent: card_arrival
Confidence: 35.02 %
------------------------------------------------------------
Customer: I forgot my PIN
Intent: pin_blocked
Confidence: 48.24 %
------------------------------------------------------------
Customer: Why was I charged for withdrawing cash?
Intent: cash_withdrawal_charge
Confidence: 73.68 %
------------------------------------------------------------
Customer: I don't recognize this transaction
Intent: compromised_card
Confidence: 10.92 %
------------------------------------------------------------
Customer: My transfer is still pending
Intent: pending_transfer
Confidence: 95.09 %
------------------------------------------------------------
Customer: I want to change my PIN
Intent: change_pin
Confidence: 95.18 %
------------------------------------------------------------


## 23. Create simple banking responses

In [ ]:
responses = {
    "card_arrival": "Your card delivery is being checked. Please check the expected delivery date in your app.",
    "card_delivery_estimate": "Standard physical cards usually take 3 to 5 business days to arrive locally, and up to 2 weeks internationally.",
    "card_not_received": "If your card has not arrived after the estimated window, please report it as lost in the app so we can ship a replacement.",
    "pin_blocked": "If your PIN is blocked due to 3 incorrect attempts, you can easily unblock it in the app under Card Settings -> Security -> Unblock PIN.",
    "change_pin": "You can change your PIN at any supported ATM by selecting 'PIN Services' after inserting your physical card.",
    "cash_withdrawal_not_recognised": "If you don't recognize a cash withdrawal, freeze your card instantly in the app and contact our fraud team immediately.",
    "cash_withdrawal_charge": "We do not charge for the first 5 ATM withdrawals per month. After that, a standard 2% fee may apply depending on your plan.",
    "pending_transfer": "Transfers can remain pending while banking systems clear them. Most local transfers clear instantly, while international ones take 1-3 business days.",
    "card_payment_not_recognised": "This card payment is not recognized. Freeze your card in the app and submit a transaction dispute form if you suspect fraud.",
    "lost_or_stolen_card": "If your card is lost or stolen, please freeze it instantly in the SmartBank app and request a replacement card.",
    "declined_cash_withdrawal": "Cash withdrawals can be declined due to insufficient funds, an incorrect PIN, reaching daily ATM limits, or your card being frozen.",
    "top_up_failed": "If your top-up failed, please check that your source card has sufficient funds, is active for online transactions, or try using another card/payment method.",
    "country_support": "SmartBank is currently supported across the US, UK, EEA countries, and Switzerland. We are constantly expanding!"
}

## 24. Create the chatbot

In [ ]:
def banking_chatbot(message):
    # Basic cleaning for greetings
    greetings = ["hi", "hello", "hey", "greetings", "good morning", "good afternoon"]
    if message.strip().lower() in greetings:
        print("🤖 SmartBank: Hello! I am your SmartBank digital assistant. How can I help you today?")
        return

    intent, confidence = predict_intent(message)

    # Smart thresholds without extra debug logging
    if confidence >= 0.45:
        response = responses.get(
            intent,
            f"I've identified your request regarding '{intent.replace('_', ' ')}'. How can I assist you further?"
        )
    elif confidence >= 0.20:
        friendly_intent = intent.replace('_', ' ')
        response = f"It sounds like you might be asking about '{friendly_intent}'. " + responses.get(intent, "Please contact our support desk.")
    else:
        response = "I'm sorry, I couldn't quite understand your request. Could you please rephrase it with more details?"

    print(f"🤖 SmartBank: {response}")

## 25. Try the chatbot

In [ ]:
banking_chatbot("I don't know where my card is")


🤖 SmartBank
Intent Detected: card_arrival (Confidence: 6.56%)
Response: I'm sorry, I couldn't quite understand your request. Could you please rephrase it or describe your issue in a bit more detail?


In [ ]:
banking_chatbot("I forgot my PIN")


🤖 SmartBank
Intent Detected: pin_blocked (Confidence: 48.24%)
Response: I've identified your request regarding 'pin blocked'. How can I assist you further with this?


In [ ]:
banking_chatbot("Someone withdrew money from my account")


🤖 SmartBank
Intent Detected: declined_cash_withdrawal (Confidence: 20.67%)
Response: I'm not entirely sure, but it sounds like you might be asking about 'declined cash withdrawal'. If so, here is some info: Please contact our support desk.


## 26. Interactive chatbot

In [ ]:
# Final Simplified Interactive Chatbot Loop using local model
print("Starting local SmartBank Chatbot. Type 'exit' to quit.")
while True:
    user_input = input("\nYou: ")
    if user_input.lower() == "exit":
        print("SmartBank: Thank you! Goodbye.")
        break
    banking_chatbot(user_input)

Starting local SmartBank Chatbot. Type 'exit' to quit.

You: hi
🤖 SmartBank: Hello! I am your SmartBank digital assistant. How can I help you today?

You: hello
🤖 SmartBank: Hello! I am your SmartBank digital assistant. How can I help you today?

You: card lost
🤖 SmartBank: It sounds like you might be asking about 'lost or stolen card'. If your card is lost or stolen, please block it immediately through your official banking channel.

You: exit
SmartBank: Thank you! Goodbye.


## Project explanation

**Problem:** Identify the intent behind a banking customer question.

**spaCy:** Cleans and lemmatizes the text.

**TF-IDF:** Converts text into numerical features.

**Logistic Regression:** Classifies the customer question into a banking intent.

**Chatbot:** Takes a new question, predicts the intent, checks confidence, and returns a simple response.

### End-to-end flow
`Customer Question → spaCy → TF-IDF → Logistic Regression → Intent → Response`

**Important:** This is a demo intent-classification system, not a real banking service. Never enter actual banking credentials or sensitive financial information.


In [ ]:
# Gemini fallback function deactivated